# Jet clustering and leading-hadron tagging of the struck quark

In deep-inelastic scattering (DIS) the virtual photon knocks a quark out of the proton. That quark
turns into a spray of hadrons, the **current jet**. In this notebook you take events that have
already been generated, cluster the current jet yourself, and use the jet's **leading hadron** to
tell a struck **u** quark from a struck **d** quark.

1. Load the events and look at one of them.
2. Cluster jets with the e⁺e⁻ anti-k<sub>T</sub> algorithm.
3. Find the leading charged hadron of the jet and check the result against the generator.
4. Measure the purity and efficiency of the leading-hadron tag as a function of a cut on z<sub>h</sub>.

**You need:** Python 3 with `numpy`, `pandas`, `matplotlib` and Jupyter (or VS Code / Cursor with
`ipykernel`).

**The events** are made once (after the Track C installation, `setup_pythia.sh` and `build.sh`) with

```bash
scripts/shower_study/tagging/make_notebook_sample.sh            # 300k events, parton showers on
scripts/shower_study/tagging/make_notebook_sample.sh 300000 0   # the same with showers off
```

They are PYTHIA 8.311 DIS events at the EIC with a 5 GeV electron on a 41 GeV proton,
16 < Q² < 25 GeV² and 0.05 < x < 0.5, all struck-quark flavours, no short-range correlations.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STUDY_ROOT = Path(os.environ.get("STUDY_ROOT", Path.home() / "Data/fig3_shower_study"))
SAMPLE = "showers_on"   # or "showers_off"
N_EVENTS = None         # None = all events (clustering 300k events takes about two minutes)
R = 1.0                 # jet radius in radians

events = pd.read_csv(STUDY_ROOT / "notebook" / f"events_{SAMPLE}.csv")
particles = pd.read_csv(STUDY_ROOT / "notebook" / f"particles_{SAMPLE}.csv")
if N_EVENTS is not None:
    events = events[events["event_id"] < N_EVENTS]
    particles = particles[particles["event_id"] < N_EVENTS]
print(f"{len(events):,} events, {len(particles):,} particles ({len(particles) / len(events):.1f} per event)")

## 1. The data

`events_*.csv` has one row per event. The columns used here are

* `struck_id`: the **true** flavour of the struck quark (PDG code: 2 = u, 1 = d, 3 = s, negative = antiquark).
  An experiment cannot see this; we use it only to measure how well a tag works.
* `Q2`, `xB`: the DIS kinematics, needed for z<sub>h</sub>.
* `tagjet_E`: the energy of the current jet the generator itself found (with FastJet), and
  `leadcur_id`, `leadcur_z`: the generator's leading charged hadron of the whole current hemisphere
  (no jet). Both are only used for checks.

`particles_*.csv` has one row per visible final-state particle (the scattered electron and neutrinos
removed): `pdg_id`, `charge` and the four-momentum `px, py, pz, e` in GeV in the **Breit frame**.
In this frame the proton moves along +z and the virtual photon along −z with zero energy. The struck
quark recoils along −z, so the current jet is in the hemisphere **pz < 0**; the proton remnant goes
to pz > 0.

In [ ]:
flavour = {2: "u", 1: "d", 3: "s", 4: "c", -2: "ubar", -1: "dbar", -3: "sbar", -4: "cbar"}
print("struck-quark flavour fractions:")
print(events["struck_id"].map(flavour).value_counts(normalize=True).round(3).to_string())

### One event

The hadronic final state carries the momentum of the proton plus the photon, which has no transverse
component in the Breit frame: the particles' p<sub>x</sub> and p<sub>y</sub> add up to (almost) zero.

In [ ]:
event0 = particles[particles["event_id"] == 0]
print(event0.to_string(index=False))
print("\nsum of px, py, pz, e:", event0[["px", "py", "pz", "e"]].sum().round(3).to_dict())

## 2. Jet clustering: e⁺e⁻ anti-k<sub>T</sub>

Sequential-recombination algorithms build jets by repeatedly merging the two "closest" objects. For
every pair of objects i, j and every single object i we compute

$$
d_{ij} = \min\left(E_i^{-2}, E_j^{-2}\right)\,\frac{1 - \cos\theta_{ij}}{1 - \cos R},
\qquad d_{iB} = E_i^{-2},
$$

where θ<sub>ij</sub> is the opening angle and R the jet radius. Then repeat:

* find the smallest of all d<sub>ij</sub> and d<sub>iB</sub>;
* if it is a d<sub>ij</sub>, replace i and j by one object with four-momentum p<sub>i</sub> + p<sub>j</sub>;
* if it is a d<sub>iB</sub>, object i is a finished jet and is removed from the list,

until nothing is left. The E<sup>−2</sup> factor makes energetic particles cluster first, so jets
grow around the hardest particles and are roughly cones of half-angle R. This is FastJet's
`ee_genkt_algorithm` with p = −1, the algorithm the generator uses. We use energies and angles
rather than transverse momenta and rapidities because the current jet points along the −z axis.

In [ ]:
def ee_antikt(p4, R=1.0):
    '''Cluster particles with e+e- anti-kT (FastJet ee_genkt, p = -1).

    p4: array of shape (n, 4) with columns px, py, pz, E.
    Returns a list of jets, each (four-momentum, list of particle indices).
    '''
    momenta = [row.copy() for row in p4]
    members = [[i] for i in range(len(p4))]
    jets = []
    while momenta:
        P = np.array(momenta)
        d_iB = 1.0 / P[:, 3] ** 2
        unit = P[:, :3] / np.linalg.norm(P[:, :3], axis=1, keepdims=True)
        one_minus_cos = 1.0 - np.clip(unit @ unit.T, -1.0, 1.0)
        d_ij = np.minimum.outer(d_iB, d_iB) * one_minus_cos / (1.0 - np.cos(R))
        np.fill_diagonal(d_ij, np.inf)
        i, j = np.unravel_index(np.argmin(d_ij), d_ij.shape)
        k = np.argmin(d_iB)
        if d_ij[i, j] < d_iB[k]:            # merge the closest pair into object i
            momenta[i] = momenta[i] + momenta[j]
            members[i] = members[i] + members[j]
            del momenta[j], members[j]
        else:                               # object k is a finished jet
            jets.append((momenta.pop(k), members.pop(k)))
    return jets


def current_jet(p4, R=1.0):
    '''The highest-energy jet in the current hemisphere (pz < 0), or None.'''
    jets = [jet for jet in ee_antikt(p4, R) if jet[0][2] < 0]
    return max(jets, key=lambda jet: jet[0][3]) if jets else None


p4 = event0[["px", "py", "pz", "e"]].to_numpy()
for momentum, idx in ee_antikt(p4, R):
    print(f"jet: E = {momentum[3]:.3f} GeV, pz = {momentum[2]:+.3f} GeV, constituents {idx}")

## 3. The leading hadron of the jet

When the struck quark fragments, the hadron carrying the largest share of its momentum most often
contains the quark itself (the "leading-particle effect"). A struck **u** tends to end up in a
π⁺ (u d̄) or K⁺ (u s̄), a struck **d** in a π⁻ (d ū). So the charge and species of the leading
hadron tag the struck flavour.

We measure a hadron's share with the SIDIS variable

$$
z_h = \frac{P\cdot p_h}{P\cdot q},
$$

the fraction of the photon's energy (in the proton rest frame) that the hadron carries. In the Breit
frame P = (E<sub>P</sub>, 0, 0, P<sub>z</sub>) with P<sub>z</sub> = Q/(2x) and q = (0, 0, 0, −Q), so

$$
z_h = \frac{E_P E_h - P_z\, p_{z,h}}{Q^2 / 2x} \;\approx\; \frac{E_h - p_{z,h}}{Q}
\quad\text{(massless proton)} .
$$

For every event: cluster the current jet, compute z<sub>h</sub> of its charged hadrons, and keep the
one with the largest z<sub>h</sub>.

In [ ]:
M_P = 0.938272   # proton mass, GeV
kinematics = events.set_index("event_id")[["Q2", "xB"]]
rows = []
for event_id, ev in particles.groupby("event_id", sort=True):
    p4 = ev[["px", "py", "pz", "e"]].to_numpy()
    jet = current_jet(p4, R)
    if jet is None:
        continue
    momentum, idx = jet
    Q2, x = kinematics.loc[event_id]
    Pz = np.sqrt(Q2) / (2 * x)                                   # proton momentum in the Breit frame
    z_h = (np.hypot(Pz, M_P) * p4[:, 3] - Pz * p4[:, 2]) / (Q2 / (2 * x))
    pdg, charge = ev["pdg_id"].to_numpy(), ev["charge"].to_numpy()
    charged_hadrons = [i for i in idx if charge[i] != 0 and abs(pdg[i]) > 100]   # no leptons
    lead = max(charged_hadrons, key=lambda i: z_h[i]) if charged_hadrons else None
    rows.append({"event_id": event_id, "jet_E": momentum[3],
                 "jet_lead_id": pdg[lead] if lead is not None else 0,
                 "jet_lead_z": z_h[lead] if lead is not None else np.nan})

lead = events.merge(pd.DataFrame(rows), on="event_id", how="left")
lead["jet_lead_id"] = lead["jet_lead_id"].fillna(0).astype(int)
print(f"{lead['jet_E'].notna().mean():.1%} of events have a current jet, "
      f"{(lead['lead_id'] != 0).mean():.1%} have a charged hadron in it")

### Check against the generator

With R = 1 our jets should be exactly the generator's. The generator also stored the leading charged
hadron of the **whole** current hemisphere (no jet): how often is that the same particle as the
leading charged hadron of our jet?

In [ ]:
has_jet = lead["jet_E"].notna()
same_jet = np.isclose(lead.loc[has_jet, "jet_E"], lead.loc[has_jet, "tagjet_E"], rtol=1e-4)
same_lead = (lead["jet_lead_id"] == lead["leadcur_id"]) & np.isclose(lead["jet_lead_z"], lead["leadcur_z"], rtol=1e-4)
print(f"jet energy agrees with the generator's jet in {same_jet.mean():.2%} of events")
print(f"the jet's leading charged hadron is the leading charged hadron of the whole current hemisphere "
      f"in {same_lead[has_jet].mean():.1%} of events")

### Which hadron leads?

Fractions of struck-u and struck-d events by the species of the jet's leading charged hadron
(no z<sub>h</sub> cut yet).

In [ ]:
u = lead[lead["struck_id"] == 2]
d = lead[lead["struck_id"] == 1]
species = {211: "pi+", -211: "pi-", 321: "K+", -321: "K-", 2212: "p", -2212: "pbar"}
table = pd.DataFrame({
    "struck u": u["jet_lead_id"].map(species).fillna("other or none").value_counts(normalize=True),
    "struck d": d["jet_lead_id"].map(species).fillna("other or none").value_counts(normalize=True),
}).fillna(0).round(3)
print(f"{len(u):,} struck-u and {len(d):,} struck-d events\n")
print(table.to_string())

## 4. Purity and efficiency of the leading-hadron tag

Tag an event as "u" when the jet's leading charged hadron is a π⁺ (or a π⁺ or K⁺, or any of π⁺, K⁺,
p) **and** its z<sub>h</sub> is above a cut. For each cut:

* **events tagged**: the fraction of all struck-u (struck-d) events that pass the tag (the efficiency);
* **u purity**: the fraction of tagged events whose struck quark is a u rather than a d,
  ε<sub>u</sub> f<sub>u</sub> / (ε<sub>u</sub> f<sub>u</sub> + ε<sub>d</sub> f<sub>d</sub>),
  with the struck-quark mix f<sub>u</sub> : f<sub>d</sub> = 0.742 : 0.116 used in our Track D plots.

A higher cut keeps a purer but smaller sample.

In [ ]:
F_U, F_D = 0.742, 0.116    # struck u : d mix for the purity
TAGGERS = [                # label, accepted PDG codes of the leading hadron, line style
    (r"leading $\pi^+$", [211], "-"),
    (r"leading $\pi^+$ or $K^+$", [211, 321], ":"),
    (r"leading positive ($\pi^+$, $K^+$, $p$)", [211, 321, 2212], "--"),
]
cuts = np.arange(0.0, 0.901, 0.025)


def tagged_fraction(df, ids, cut, id_col="jet_lead_id", z_col="jet_lead_z"):
    '''Fraction of the events in df whose leading hadron is one of ids with z_h > cut.'''
    return (df[id_col].isin(ids) & (df[z_col] > cut)).mean()


fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for label, ids, ls in TAGGERS:
    eff_u = np.array([tagged_fraction(u, ids, c) for c in cuts])
    eff_d = np.array([tagged_fraction(d, ids, c) for c in cuts])
    purity = F_U * eff_u / (F_U * eff_u + F_D * eff_d)
    ax[0].plot(cuts, purity, color="k", ls=ls, lw=2, label=label)
    ax[1].plot(cuts, 100 * eff_u, color="tab:red", ls=ls, lw=2)
    ax[1].plot(cuts, 100 * eff_d, color="tab:blue", ls=ls, lw=2)
ax[0].axhline(F_U / (F_U + F_D), color="0.5", ls=":", label="no tag")
ax[1].plot([], [], color="tab:red", lw=2, label="struck $u$")
ax[1].plot([], [], color="tab:blue", lw=2, label="struck $d$")
ax[0].set_ylabel("$u$ purity of the tag")
ax[1].set_ylabel("events tagged [%]")
ax[0].legend(frameon=False, loc="lower right")
ax[1].legend(frameon=False, loc="upper right")
for a in ax:
    a.set_xlabel("cut on $z_h$ of the leading charged hadron of the jet")
    a.set_xlim(0, 0.9)
    a.tick_params(which="both", direction="in")
ax[1].set_ylim(bottom=0)
plt.tight_layout()
plt.show()

### Jet vs whole hemisphere

The same numbers when the leading hadron is taken from the whole current hemisphere instead of the
jet (the generator's `leadcur_id`, `leadcur_z`), at two cuts.

In [ ]:
print("events tagged, struck u / struck d, and u purity")
for label, ids, _ in TAGGERS:
    name = label.replace("$", "").replace("^", "").replace("\\", "")
    for cut in (0.0, 0.5):
        parts = []
        for id_col, z_col, where in (("jet_lead_id", "jet_lead_z", "jet"), ("leadcur_id", "leadcur_z", "hemisphere")):
            eu, ed = tagged_fraction(u, ids, cut, id_col, z_col), tagged_fraction(d, ids, cut, id_col, z_col)
            parts.append(f"{where}: {eu:.3f} / {ed:.3f}, purity {F_U * eu / (F_U * eu + F_D * ed):.3f}")
        print(f"  {name:32s} z_h > {cut}:   " + "    ".join(parts))

## Things to try

1. Change the jet radius `R` (for example 0.5 or 1.5). How often does the jet still contain the
   hemisphere's leading hadron, and how do the tag efficiencies change?
2. Pick the leading hadron by **energy** in the Breit frame instead of z<sub>h</sub>. Does the tag change?
3. Make a **d-tag**: the leading charged hadron is a π⁻ with z<sub>h</sub> above a cut. Why is its purity
   so much lower? (Look at the flavour fractions in section 1.)
4. Make the showers-off sample and set `SAMPLE = "showers_off"`. Does parton showering change the tag?
5. Use the sample's own struck-u and struck-d fractions instead of 0.742 : 0.116 for the purity.